<a href="https://colab.research.google.com/github/TediBalint/AI-Jegyzetek/blob/master/Computer%20Vision/Bevezet%C3%A9s%20a%20GANokba.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bevezetés a GAN-okba (Generative Adversarial Networks)

A **GAN** (Generative Adversarial Network) egy generatív modell, amely két neurális háló **versengésén** alapul:

- **Generator (G)**: Véletlenszerű zajból próbál valósághű adatot (pl. képet) generálni
- **Discriminator (D)**: Megpróbálja megkülönböztetni a valódi és generált adatokat

Ez egy **kétjátékos minimax játék**: G minimalizálni, D maximalizálni próbálja ugyanazt a célfüggvényt.

## Tartalomjegyzék

1. GAN alapelv és matematika
2. Egyszerű GAN 2D adatokon
3. DCGAN képgeneráláshoz
4. Tanítási trükkök és problémák
5. GAN variánsok

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'DejaVu Sans'
torch.manual_seed(42)
np.random.seed(42)

## 1. GAN alapelv és matematika

### A minimax célfüggvény

$$\min_G \max_D V(D, G) = \mathbb{E}_{x \sim p_{data}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$$

**Mit jelentenek a tagok?**

| Tag | Jelentés | Ki optimalizálja? |
|-----|----------|-------------------|
| $\log D(x)$ | D mennyire ismeri fel a valódi adatot | D maximalizálja |
| $\log(1 - D(G(z)))$ | D mennyire ismeri fel a hamisítványt | D max., G min. |

### Az edzési folyamat

```
┌─────────────────────────────────────────────────────────┐
│  Zaj (z)  ──▶  Generator  ──▶  Hamis kép               │
│                                    │                    │
│                                    ▼                    │
│  Valódi kép ─────────────────▶ Discriminator ──▶ 0/1   │
└─────────────────────────────────────────────────────────┘
```

**Nash-egyensúly**: Ideális esetben D nem tud különbséget tenni → $D(x) = 0.5$ mindenre.

In [ ]:
# GAN komponensek vizualizálása
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 1. Latent tér
z = np.random.randn(500, 2)
axes[0].scatter(z[:, 0], z[:, 1], alpha=0.5, s=10)
axes[0].set_title('Latent tér (zaj)\n$z \\sim \\mathcal{N}(0, I)$', fontsize=11)
axes[0].set_xlabel('$z_1$')
axes[0].set_ylabel('$z_2$')

# 2. Cél eloszlás (amit G-nek meg kell tanulnia)
theta = np.random.uniform(0, 2*np.pi, 500)
r = 2 + np.random.normal(0, 0.15, 500)
x_real = np.stack([r * np.cos(theta), r * np.sin(theta)], axis=1)
axes[1].scatter(x_real[:, 0], x_real[:, 1], alpha=0.5, s=10, c='green')
axes[1].set_title('Cél eloszlás\n$p_{data}$ (kör)', fontsize=11)
axes[1].set_xlim(-4, 4)
axes[1].set_ylim(-4, 4)

# 3. Tanítás előtt (G random transzformáció)
W = np.random.randn(2, 2) * 0.5
x_fake = z @ W
axes[2].scatter(x_fake[:, 0], x_fake[:, 1], alpha=0.5, s=10, c='red', label='Generált')
axes[2].scatter(x_real[:, 0], x_real[:, 1], alpha=0.3, s=10, c='green', label='Valódi')
axes[2].set_title('Tanítás előtt\nG(z) vs valódi', fontsize=11)
axes[2].legend()
axes[2].set_xlim(-4, 4)
axes[2].set_ylim(-4, 4)

plt.tight_layout()
plt.show()

## 2. Egyszerű GAN 2D adatokon

Először egy egyszerű példán mutatjuk be: a Generator megtanulja egy kör alakú eloszlás generálását.

In [ ]:
# Hálózatok definíciója
latent_dim = 2
data_dim = 2

class Generator(nn.Module):
    """Zaj → Generált adat transzformáció."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, data_dim)
        )

    def forward(self, z):
        return self.net(z)

class Discriminator(nn.Module):
    """Adat → Valódi/Hamis valószínűség."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(data_dim, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

# Segédfüggvények
def sample_real(n):
    """Valódi adatok mintavételezése (kör eloszlás)."""
    theta = np.random.uniform(0, 2*np.pi, n)
    r = 2 + np.random.normal(0, 0.1, n)
    x = r * np.cos(theta)
    y = r * np.sin(theta)
    return torch.FloatTensor(np.stack([x, y], axis=1))

def sample_noise(n):
    """Zaj mintavételezése a latent térből."""
    return torch.randn(n, latent_dim)

In [ ]:
# GAN tanítás
G = Generator()
D = Discriminator()

criterion = nn.BCELoss()
opt_G = optim.Adam(G.parameters(), lr=0.0002, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=0.0002, betas=(0.5, 0.999))

n_epochs = 3000
batch_size = 256
history = {'d_loss': [], 'g_loss': [], 'd_real': [], 'd_fake': []}
snapshots = []  # Generált minták mentése animációhoz

for epoch in range(n_epochs):
    # === Discriminator tanítás ===
    real_data = sample_real(batch_size)
    fake_data = G(sample_noise(batch_size)).detach()

    real_labels = torch.ones(batch_size, 1)
    fake_labels = torch.zeros(batch_size, 1)

    opt_D.zero_grad()
    d_real = D(real_data)
    d_fake = D(fake_data)
    d_loss = criterion(d_real, real_labels) + criterion(d_fake, fake_labels)
    d_loss.backward()
    opt_D.step()

    # === Generator tanítás ===
    opt_G.zero_grad()
    fake_data = G(sample_noise(batch_size))
    g_loss = criterion(D(fake_data), real_labels)  # G azt akarja, hogy D valódinak lássa
    g_loss.backward()
    opt_G.step()

    # Logolás
    history['d_loss'].append(d_loss.item())
    history['g_loss'].append(g_loss.item())
    history['d_real'].append(d_real.mean().item())
    history['d_fake'].append(d_fake.mean().item())

    if epoch % 100 == 0:
        with torch.no_grad():
            snapshots.append(G(sample_noise(500)).numpy())

print(f"Tanítás kész! Végső D loss: {history['d_loss'][-1]:.4f}, G loss: {history['g_loss'][-1]:.4f}")

In [ ]:
# Eredmények vizualizálása
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. Loss görbék
axes[0, 0].plot(history['d_loss'], label='D loss', alpha=0.7)
axes[0, 0].plot(history['g_loss'], label='G loss', alpha=0.7)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].set_title('Tanítási görbék')

# 2. D kimenetek
axes[0, 1].plot(history['d_real'], label='D(valódi)', alpha=0.7)
axes[0, 1].plot(history['d_fake'], label='D(generált)', alpha=0.7)
axes[0, 1].axhline(0.5, color='gray', linestyle='--', label='Ideális')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('D kimenet')
axes[0, 1].legend()
axes[0, 1].set_title('Discriminator kimenetek\n(0.5 felé konvergál = egyensúly)')

# 3. Valós vs Generált eloszlás
real = sample_real(1000).numpy()
with torch.no_grad():
    fake = G(sample_noise(1000)).numpy()

axes[1, 0].scatter(real[:, 0], real[:, 1], alpha=0.4, s=15, label='Valódi', c='green')
axes[1, 0].scatter(fake[:, 0], fake[:, 1], alpha=0.4, s=15, label='Generált', c='red')
axes[1, 0].legend()
axes[1, 0].set_title('Végső eloszlások')
axes[1, 0].set_xlim(-4, 4)
axes[1, 0].set_ylim(-4, 4)
axes[1, 0].set_aspect('equal')

# 4. Discriminator döntési felület
xx, yy = np.meshgrid(np.linspace(-4, 4, 100), np.linspace(-4, 4, 100))
grid = torch.FloatTensor(np.c_[xx.ravel(), yy.ravel()])
with torch.no_grad():
    Z = D(grid).numpy().reshape(xx.shape)

contour = axes[1, 1].contourf(xx, yy, Z, levels=20, cmap='RdBu_r', alpha=0.8)
axes[1, 1].scatter(real[:, 0], real[:, 1], c='green', s=5, alpha=0.3)
plt.colorbar(contour, ax=axes[1, 1], label='D(x)')
axes[1, 1].set_title('Discriminator döntési felület\n(piros=valódi, kék=hamis)')

plt.tight_layout()
plt.show()

In [ ]:
# Tanulási folyamat animáció (snapshots)
fig, ax = plt.subplots(figsize=(6, 6))
real = sample_real(500).numpy()

def animate(i):
    ax.clear()
    ax.scatter(real[:, 0], real[:, 1], alpha=0.3, s=10, c='green', label='Valódi')
    ax.scatter(snapshots[i][:, 0], snapshots[i][:, 1], alpha=0.5, s=10, c='red', label='Generált')
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_title(f'Epoch: {i * 100}')
    ax.legend()
    ax.set_aspect('equal')

anim = FuncAnimation(fig, animate, frames=len(snapshots), interval=150)
plt.close()
HTML(anim.to_jshtml())

## 3. DCGAN képgeneráláshoz

A **DCGAN** (Deep Convolutional GAN) konvolúciós rétegeket használ képek generálásához.

### Architektúra szabályok (Radford et al., 2015)

1. Pooling helyett strided konvolúció (D) / transposed konvolúció (G)
2. BatchNorm mindkét hálózatban (kivéve G output és D input)
3. Nincs fully connected réteg (kivéve G bemenete)
4. G: ReLU aktiváció (output: Tanh), D: LeakyReLU

In [ ]:
class DCGANGenerator(nn.Module):
    """DCGAN Generator 28×28 képekhez (pl. MNIST)."""
    def __init__(self, latent_dim=100, channels=1):
        super().__init__()
        self.latent_dim = latent_dim

        self.net = nn.Sequential(
            # Latent → 7×7×256
            nn.Linear(latent_dim, 256 * 7 * 7),
            nn.BatchNorm1d(256 * 7 * 7),
            nn.ReLU(True),
            nn.Unflatten(1, (256, 7, 7)),

            # 7×7 → 14×14
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            # 14×14 → 28×28
            nn.ConvTranspose2d(128, channels, kernel_size=4, stride=2, padding=1),
            nn.Tanh()  # Output: [-1, 1]
        )

    def forward(self, z):
        return self.net(z)

class DCGANDiscriminator(nn.Module):
    """DCGAN Discriminator 28×28 képekhez."""
    def __init__(self, channels=1):
        super().__init__()

        self.net = nn.Sequential(
            # 28×28 → 14×14
            nn.Conv2d(channels, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),

            # 14×14 → 7×7
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            # 7×7 → 1
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

# Architektúra teszt
g = DCGANGenerator(latent_dim=100)
d = DCGANDiscriminator()

z = torch.randn(4, 100)
fake_img = g(z)
pred = d(fake_img)

print(f"Generator: z {z.shape} → img {fake_img.shape}")
print(f"Discriminator: img {fake_img.shape} → pred {pred.shape}")

In [ ]:
# Generált "képek" (tanítás nélkül - random)
fig, axes = plt.subplots(2, 8, figsize=(14, 4))

with torch.no_grad():
    z = torch.randn(16, 100)
    fake_images = g(z)

for i, ax in enumerate(axes.flat):
    img = fake_images[i, 0].numpy()
    ax.imshow(img, cmap='gray')
    ax.axis('off')

plt.suptitle('DCGAN Generator kimenet (tanítás NÉLKÜL - random zaj)', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Tanítási trükkök és problémák

### Gyakori problémák

| Probléma | Tünet | Megoldás |
|----------|-------|----------|
| **Mode collapse** | G mindig ugyanazt generálja | Minibatch discrimination, unrolled GAN |
| **Vanishing gradient** | D túl jó, G nem tanul | Label smoothing, Wasserstein loss |
| **Instabil tanítás** | Loss oszcillál | Spectral normalization, learning rate csökkentés |
| **D/G egyensúly** | Egyik dominál | D-t többször tanítjuk, vagy G-t |

### Bevált trükkök

In [ ]:
# 1. Label smoothing
def get_labels_smooth(batch_size, real=True, smoothing=0.1):
    """Soft labels: 1.0 → 0.9, 0.0 → 0.1"""
    if real:
        return torch.ones(batch_size, 1) * (1 - smoothing)
    else:
        return torch.ones(batch_size, 1) * smoothing

# 2. Noise injection a D bemenetére
def add_instance_noise(x, std=0.1):
    """Gaussi zaj hozzáadása - segít stabilizálni."""
    return x + torch.randn_like(x) * std

# 3. Spectral normalization (PyTorch beépített)
# nn.utils.spectral_norm(layer)

print("Label smoothing példa:")
print(f"  Real labels: {get_labels_smooth(3, real=True).flatten().tolist()}")
print(f"  Fake labels: {get_labels_smooth(3, real=False).flatten().tolist()}")

In [ ]:
# Mode collapse demonstráció
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Normál működés
good_samples = np.random.randn(500, 2)
good_samples = good_samples / np.linalg.norm(good_samples, axis=1, keepdims=True) * 2
axes[0].scatter(good_samples[:, 0], good_samples[:, 1], alpha=0.5, s=10)
axes[0].set_title('Jó: Teljes eloszlás', fontsize=11)
axes[0].set_xlim(-3, 3)
axes[0].set_ylim(-3, 3)

# Partial mode collapse
partial = np.concatenate([
    np.random.randn(200, 2) * 0.2 + [2, 0],
    np.random.randn(200, 2) * 0.2 + [-2, 0]
])
axes[1].scatter(partial[:, 0], partial[:, 1], alpha=0.5, s=10, c='orange')
axes[1].set_title('Részleges mode collapse\n(csak néhány mód)', fontsize=11)
axes[1].set_xlim(-3, 3)
axes[1].set_ylim(-3, 3)

# Full mode collapse
collapsed = np.random.randn(500, 2) * 0.1 + [1, 1]
axes[2].scatter(collapsed[:, 0], collapsed[:, 1], alpha=0.5, s=10, c='red')
axes[2].set_title('Teljes mode collapse\n(mindig ugyanaz)', fontsize=11)
axes[2].set_xlim(-3, 3)
axes[2].set_ylim(-3, 3)

plt.tight_layout()
plt.show()

## 5. GAN variánsok

| Variáns | Év | Fő újítás | Alkalmazás |
|---------|-----|-----------|------------|
| **DCGAN** | 2015 | Konvolúciós architektúra | Képgenerálás |
| **WGAN** | 2017 | Wasserstein távolság | Stabilabb tanítás |
| **CGAN** | 2014 | Feltételes generálás | Irányított generálás |
| **CycleGAN** | 2017 | Párosítatlan fordítás | Stílus transzfer |
| **StyleGAN** | 2019 | Stílus-alapú generálás | Fotorealisztikus arcok |
| **BigGAN** | 2018 | Nagy batch, class-conditional | ImageNet generálás |

In [ ]:
# Conditional GAN (cGAN) koncepció
class ConditionalGenerator(nn.Module):
    """Feltételes Generator - osztálycímke is bemenet."""
    def __init__(self, latent_dim=100, n_classes=10, embed_dim=50):
        super().__init__()
        self.label_embed = nn.Embedding(n_classes, embed_dim)

        self.net = nn.Sequential(
            nn.Linear(latent_dim + embed_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 784),  # 28×28
            nn.Tanh()
        )

    def forward(self, z, labels):
        # Címke beágyazás
        label_embed = self.label_embed(labels)
        # Összefűzés
        x = torch.cat([z, label_embed], dim=1)
        return self.net(x).view(-1, 1, 28, 28)

# Teszt
cgan = ConditionalGenerator()
z = torch.randn(4, 100)
labels = torch.tensor([0, 3, 5, 7])  # "Generálj 0-t, 3-at, 5-öt, 7-et"
output = cgan(z, labels)
print(f"Conditional GAN: z {z.shape} + labels {labels.shape} → {output.shape}")

## Összefoglalás

### GAN lényege
- **Két hálózat** verseng: Generator vs Discriminator
- **Minimax játék** → Nash-egyensúly
- G megtanulja az adateloszlást, D nem tud különbséget tenni

### Architektúra
- **Generator**: Latent zaj → Generált kép (upsampling)
- **Discriminator**: Kép → Valós/Hamis valószínűség (downsampling)

### Kulcs tanulságok
1. A GAN tanítás instabil - sok trükk kell
2. Mode collapse gyakori probléma
3. Modern variánsok (StyleGAN, Diffusion) már megbízhatóbbak

### Kód a gyakorlatban
```python
# Egyszerű GAN tanítási lépés
# 1. D tanítás valódi adaton
d_loss_real = criterion(D(real_data), ones)
# 2. D tanítás hamis adaton  
d_loss_fake = criterion(D(G(z).detach()), zeros)
# 3. G tanítás - D-t megtéveszteni
g_loss = criterion(D(G(z)), ones)
```